# 02 · 描述统计与 MLE 直觉 Descriptive Statistics & MLE Intuition

> **能力标签**：SH6（概率与统计 / Probability & statistics）

## 目标 Objectives

完成本课后，你应该能够：

1. 计算**描述统计量**：均值（mean）、方差（variance，总体 ddof=0）、标准差（std）、中位数（median）。
2. 理解**最大似然估计**（MLE）直觉：正态分布均值的 MLE 就是样本均值。
3. 区分**偏差**（bias）与**方差**（variance）在估计量中的含义。

> 对应能力：**SH6**


## 概念讲解 Concepts

### 描述统计量 Descriptive Statistics

描述统计量是对数据集分布形态的数值摘要：

| 统计量 | 定义 | NumPy |
|--------|------|-------|
| **均值** (mean) | $\bar{x} = \frac{1}{n}\sum_{i=1}^n x_i$ | `x.mean()` |
| **方差** (variance) | $\sigma^2 = \frac{1}{n}\sum_{i=1}^n (x_i - \bar{x})^2$ | `x.var(ddof=0)` |
| **标准差** (std) | $\sigma = \sqrt{\sigma^2}$ | `x.std(ddof=0)` |
| **中位数** (median) | 排序后中间值 | `np.median(x)` |

> **注意**：`ddof=0` 为**总体**方差（除以 $n$）；`ddof=1` 为**样本**方差（除以 $n-1$，无偏估计）。

---

### 最大似然估计 MLE 直觉

**最大似然估计**（Maximum Likelihood Estimation）：选取参数 $\theta$ 使得观测数据的联合概率最大。

对于 $X_1, \ldots, X_n \sim \mathcal{N}(\mu, \sigma^2)$：

$$\hat{\mu}_{\text{MLE}} = \arg\max_\mu \prod_{i=1}^n f(x_i; \mu, \sigma^2) = \bar{x}$$

直觉：正态分布关于均值对称，样本均值是使似然函数取极大值的 $\mu$ 估计，这就是**样本均值即 MLE**。

---

### 偏差-方差 Bias-Variance

对于估计量 $\hat{\theta}$：

- **偏差**（bias）：$\text{Bias}(\hat{\theta}) = E[\hat{\theta}] - \theta$
  反映系统误差（是否平均而言准确？）
- **方差**（variance）：$\text{Var}(\hat{\theta}) = E[(\hat{\theta} - E[\hat{\theta}])^2]$
  反映估计的稳定性（重复采样是否稳定？）

**总体方差** $s^2_{\text{pop}} = \frac{1}{n}\sum(x_i - \bar{x})^2$：对 $\sigma^2$ 有偏（低估）。
**样本方差** $s^2_{\text{sample}} = \frac{1}{n-1}\sum(x_i - \bar{x})^2$：对 $\sigma^2$ 无偏（期望 $= \sigma^2$）。


## 示例 Worked Example

In [ ]:
import numpy as np

def describe(x) -> dict:
    """返回均值、方差(总体, ddof=0)、标准差、中位数。"""
    x = np.asarray(x, dtype=float)
    return {
        "mean":   float(x.mean()),
        "var":    float(x.var()),          # ddof=0 总体方差
        "std":    float(x.std()),          # ddof=0
        "median": float(np.median(x)),
    }

# 生成正态样本
rng = np.random.default_rng(42)
data = rng.normal(loc=10.0, scale=2.0, size=500)

stats = describe(data)
print("=== 描述统计量 ===")
for k, v in stats.items():
    print(f"  {k:8s}: {v:.4f}")

# MLE 直觉演示：样本均值 = MLE(μ)
print(f"\n=== MLE 直觉 ===")
print(f"真实 μ     = 10.0000")
print(f"样本均值   = {data.mean():.4f}  ← MLE(μ)")
print(f"两者接近✓")


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

# 偏差-方差：总体方差 vs 样本方差
rng = np.random.default_rng(0)
true_var = 4.0   # σ² = 4 (σ=2)
n_reps = 5000
n_size = 10

pop_vars    = []  # ddof=0
sample_vars = []  # ddof=1

for _ in range(n_reps):
    x = rng.normal(0, 2, n_size)
    pop_vars.append(x.var(ddof=0))
    sample_vars.append(x.var(ddof=1))

pop_vars    = np.array(pop_vars)
sample_vars = np.array(sample_vars)

print("=== 偏差分析 (n=10, σ²=4, reps=5000) ===")
print(f"总体方差(ddof=0) 均值 = {pop_vars.mean():.4f}  → 偏差 = {pop_vars.mean()-true_var:+.4f}（有偏, 低估）")
print(f"样本方差(ddof=1) 均值 = {sample_vars.mean():.4f} → 偏差 = {sample_vars.mean()-true_var:+.4f}（无偏）")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, vals, label, color in zip(
    axes,
    [pop_vars, sample_vars],
    ["总体方差 ddof=0 (有偏)", "样本方差 ddof=1 (无偏)"],
    ["steelblue", "darkorange"]
):
    ax.hist(vals, bins=40, density=True, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(vals.mean(), color="red",   ls="--", lw=1.5, label=f"均值={vals.mean():.3f}")
    ax.axvline(true_var,    color="black", ls=":",  lw=1.5, label=f"真实σ²={true_var}")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("方差估计值")
    ax.legend(fontsize=8)
fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "bias_variance_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("\n图像已保存到:", _outpath)


## 动手 Exercise

In [ ]:
import numpy as np

# 练习：实现 describe() 并验证 MLE 直觉

def my_describe(x) -> dict:
    """计算均值、总体方差(ddof=0)、标准差、中位数。"""
    x = np.asarray(x, dtype=float)
    mean   = x.mean()
    var    = x.var()          # ddof=0
    std    = x.std()          # ddof=0
    median = float(np.median(x))
    return {"mean": float(mean), "var": float(var), "std": float(std), "median": median}

# 测试
rng = np.random.default_rng(7)
x = rng.normal(5, 3, 200)
result = my_describe(x)
print("描述统计量:", result)

# 验证方差公式：手动计算 = numpy 结果
manual_var = np.mean((x - x.mean())**2)
print(f"\n手动计算方差 = {manual_var:.6f}")
print(f"np.var(ddof=0) = {x.var():.6f}")
print(f"一致? {np.isclose(manual_var, x.var())}")

# MLE: 正态均值 MLE = 样本均值
print(f"\n样本均值(MLE) = {x.mean():.4f} ≈ 真实 μ=5.0")


## 延伸阅读 Further Reading

1. **《统计学习基础》（Hastie et al.）**：第 2 章 — 监督学习概述（含偏差-方差分解）
2. **Maximum Likelihood Estimation — StatQuest**（YouTube）
3. **《All of Statistics》 — Larry Wasserman**：第 6 章（MLE）
4. **NumPy 统计函数文档**：<https://numpy.org/doc/stable/reference/routines.statistics.html>


## 关联练习 Related Assignments

本课对应练习包：**`w4-descriptive`**（实现 `describe()` 函数）

```bash
python check.py w4-descriptive
```

> 能力标签：**SH6** · threshold ≥ 0.7
